# IOAI — 2025 Individual Contest Antique (Colab 자동 설정판)

이 노트북은 IOAI 로컬 연습 사이트에서 **데이터·학습환경이 자동 준비**되도록 생성되었습니다.
아래 **설정 셀을 먼저 실행**하면 공식 GitHub 저장소에서 이 문제 폴더만 부분 클론으로 받아
(전체 6.6GB 가 아니라 해당 폴더만), 그 폴더로 이동한 뒤 이후 셀이 그대로 학습/예측을 합니다.
완료 후 생성되는 제출 파일을 내려받아 연습 사이트의 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** 로 바꾸면 학습이 빨라집니다.

In [ ]:
# === 데이터 + 환경 자동 설정 (가장 먼저 실행) ===
# 공식 공개 저장소에서 이 문제 폴더만 부분 클론(sparse)으로 받고 그 폴더로 이동한다.
import os
REPO_URL = "https://github.com/IOAI-official/IOAI-2025"
CLONE = "IOAI-2025"
SUBDIR = "Individual-Contest/Antique"
# Colab 은 /content 가 홈. 재실행해도 경로가 안정적이도록 고정 기준에서 시작한다.
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
os.chdir(BASE)
if not os.path.isdir(os.path.join(CLONE, SUBDIR)):
    !git clone --filter=blob:none --no-checkout --depth 1 $REPO_URL $CLONE
    !cd $CLONE && git sparse-checkout set "$SUBDIR"
    !cd $CLONE && git checkout
os.chdir(os.path.join(BASE, CLONE, SUBDIR))
print("작업 폴더:", os.getcwd())
print("내용:", sorted(os.listdir(".")))

<img src="./figs/IOAI-Logo.png" alt="IOAI Logo" width="200" height="auto">

[IOAI 2025 (Beijing, China), Individual Contest](https://ioai-official.org/china-2025)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IOAI-official/IOAI-2025/blob/main/Individual-Contest/Antique/Antique.ipynb)

# Antique Painting Authentication

## 1. Problem Description

You have studied Artificial Intelligence for quite some time. Old friend of your father, famous archeologist and art critic, heard about this and asked for your help. You need to design an algorithm that can classify antique paintings as either authentic or replica pieces.

Because professional authentication is expensive, the research team has only obtained authenticity labels for a small portion of the paintings. For the majority of samples, the authenticity remains unknown. It is known that the paintings' digital features exhibit strong structural patterns. You are tasked with leveraging all available samples — including those with unknown labels — to train a model for classifying the authenticity of antique paintings.

## 2. Dataset

The dataset consists of a training set, a validation set and a test set, each of them has 500 independent samples. 

1. **Training Set (`training_set.csv`)**:

   - The first five columns represent the digital features of each antique painting.
   - The sixth column contains the label: 1 for authentic, -1 for replica, and 0 for unknown.

   The training set is used for training your models and can be accessed and downloaded directly during the competition.

2. **Validation Set (`validation_set.csv`)**: 
   - These are similar to the training set format but do not contain the label column.

   The validation set is used to calculate the Leaderboard A score and is not directly accessible during the competition.

3. **Test Set (`test_set.csv`)**: 
   - These are similar to the training set format but do not contain the label column.

   The test set is used to calculate the Leaderboard B score and is not directly accessible during the competition.

## 3. Task

Your task is to train an appropriate model capable of predicting the authenticity of paintings in the test sets, despite the large number of unlabeled samples.

## 4. Submission

Contestants need to submit a notebook file named `submission.ipynb`. The file should output a zip file named `submission.zip`, which should contain the following two files:

1. `submissionA.csv`: Contains the model's predicted label results on the validation set, with each line being a -1 or 1 and no header.
2. `submissionB.csv`: Contains the model's predicted label results on the test set, with each line being a -1 or 1 and no header.

The testing machine will read `submission.zip` and calculate the scores. The submission files must strictly follow the above format and naming; otherwise, the system will not be able to read them correctly. 

Details about the submission procedure are provided in the baseline notebook. Contestants are encouraged to refer to it for guidance.

## 5. Score

The evaluation metric will be **classification accuracy**, defined as the proportion of correctly predicted samples over the total number of evaluated samples.

## 6. Baseline and Training Set

- Below you can find the baseline solution.
- The dataset is in `training_set` folder.
- The highest score by the Scientific Committee for this task is 0.98 in Leaderboard B,  this score is used for score unification.
- The baseline score by the Scientific Committee for this task is 0.46 in Leaderboard B, this score is used for score unification.

### Train Your Model

In [ ]:
import os
import sys

# 1. Get the current working directory
current_dir = os.getcwd()

# 2. Check if the path contains "Individual-Contest/Antique" and trim it to that point
project_root = current_dir

# 3. Change working directory to the project root
os.chdir(project_root)
print("Working directory set to:", os.getcwd())

# 4. Add module search path (e.g., where metrics.py is located)
sys.path.append(os.path.join(project_root, "Scoring"))

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.svm import SVC

TRAIN_PATH = "./training_set/"
train = pd.read_csv(TRAIN_PATH + "training_set.csv")

X = np.array(train.iloc[:,:5])
y = np.array(train.iloc[:,5])

np.random.seed(42)
y[y == 0] = np.random.choice([-1, 1], size=(y == 0).sum())

svm_binary_model = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
svm_binary_model.fit(X, y)

### Make Predictions on the Validation and Test Set

In [ ]:
VAL_DATA_PATH = "./Solution/validation_set/"
TEST_DATA_PATH = "./Solution/test_set/"

testA = np.array(pd.read_csv(VAL_DATA_PATH + "validation_set.csv"))
testB = np.array(pd.read_csv(TEST_DATA_PATH + "test_set.csv"))

predA = svm_binary_model.predict(testA)
predB = svm_binary_model.predict(testB)

### Generate `submission.zip` for Submission

In [ ]:
import zipfile
import os

submissionA = pd.DataFrame(predA)
submissionA.to_csv("./Scoring/submissionA.csv", index=False, header=False)

submissionB = pd.DataFrame(predB)
submissionB.to_csv("./Scoring/submissionB.csv", index=False, header=False)

files_to_zip = ['./Scoring/submissionA.csv', './Scoring/submissionB.csv']
zip_filename = './Scoring/submission.zip'

with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_zip:
        zipf.write(file, os.path.basename(file))

print(f'{zip_filename} is created succefully!')

### Evaluate the Model Performance

In [ ]:
%run Scoring/metrics.py

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.zip']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)